# Mobile Battery Drain Simulation
**Course:** Simulation & Modeling Lab  **Tool:** Python (Google Colab)

## 1. Problem
"Will my phone battery last the whole day?" The answer depends on **how** you use your phone: watching videos, gaming, scrolling social media or leaving it idle.
We simulate a phone's battery minute by minute, with **random app usage**, to estimate how long a full charge lasts.

## 2. Objectives
1. Model battery drain from random phone activities.
2. Plot the battery % over one day.
3. Use **Monte Carlo simulation** (1000 days) to estimate average battery life and the **probability the battery dies before night**.
4. Compare 3 types of users (Light, Student, Gamer).
5. Test whether **Battery Saver mode** really helps (t-test).

## 3. Model & Assumptions
* The phone starts at **100%** at 8:00 AM.
* Every **10 minutes**, the user randomly picks an activity (based on their usage profile).
* Each activity drains a fixed % per minute, plus small random noise (±15%).
* The simulation stops when the battery reaches **0%**.

| Activity | Drain (% per minute) |
|---|---|
| Idle (screen off) | 0.03 |
| Calls | 0.12 |
| Social media | 0.18 |
| Video | 0.22 |
| Gaming | 0.40 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

ACTIVITIES = ["Idle", "Calls", "Social", "Video", "Gaming"]
DRAIN = {"Idle": 0.03, "Calls": 0.12, "Social": 0.18, "Video": 0.22, "Gaming": 0.40}   # % per minute

# Probability of each activity for each kind of user (each row sums to 1)
PROFILES = {
    "Light user": [0.70, 0.10, 0.15, 0.05, 0.00],
    "Student":    [0.45, 0.05, 0.30, 0.15, 0.05],
    "Gamer":      [0.35, 0.05, 0.15, 0.15, 0.30],
}
BLOCK = 10          # minutes per activity block
START_HOUR = 8      # phone unplugged at 8:00 AM
print("Setup complete")

## 4. Simulation function
`simulate_day()` runs one day and returns the battery life in hours, plus the battery level and activity for every minute.

`saver=True` turns on **Battery Saver** below 20%, which cuts the drain by 30%.

In [ ]:
def simulate_day(profile="Student", saver=False, seed=None):
    rng = np.random.default_rng(seed)
    probs = PROFILES[profile]
    battery = 100.0
    levels, acts = [battery], []
    minute = 0

    while battery > 0:
        activity = rng.choice(ACTIVITIES, p=probs)        # pick activity for this block
        for _ in range(BLOCK):
            drain = DRAIN[activity] * rng.uniform(0.85, 1.15)   # ±15% noise
            if saver and battery < 20:
                drain *= 0.7                                     # battery saver mode
            battery = max(battery - drain, 0)
            minute += 1
            levels.append(battery)
            acts.append(activity)
            if battery == 0:
                break

    return {"hours": minute / 60, "levels": np.array(levels), "acts": acts}


day = simulate_day("Student", seed=1)
dead = START_HOUR + day["hours"]
print(f"Battery lasted {day['hours']:.1f} hours -> phone dies at about {int(dead) % 24:02d}:{int((dead % 1) * 60):02d}")

## 5. Battery level during one day
The colour bar underneath shows which activity was going on at each moment. Steeper drops happen during gaming and video.

In [ ]:
colors = {"Idle": "lightgray", "Calls": "gold", "Social": "deepskyblue", "Video": "orange", "Gaming": "crimson"}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True,
                               gridspec_kw={"height_ratios": [4, 1]})
t = START_HOUR + np.arange(len(day["levels"])) / 60
ax1.plot(t, day["levels"], lw=2, color="green")
ax1.axhline(20, ls="--", color="red", label="20% low battery")
ax1.set_ylabel("battery (%)"); ax1.set_title("One simulated day — Student")
ax1.grid(alpha=0.3); ax1.legend()

for i, a in enumerate(day["acts"]):
    ax2.axvspan(t[i], t[i] + 1/60, color=colors[a])
ax2.set_yticks([]); ax2.set_xlabel("time of day (hour)")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()]
ax2.legend(handles, colors.keys(), ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.6))
plt.tight_layout(); plt.show()

## 6. Monte Carlo — 1000 days
One day is random, so we simulate **1000 days** and look at the distribution.
* **Mean battery life** with a 95% confidence interval: $\bar{x} \pm 1.96\,\dfrac{s}{\sqrt{n}}$
* **P(battery dies before 10 PM)**, i.e. lasts less than 14 hours from 8 AM

In [ ]:
N = 1000
student = np.array([simulate_day("Student", seed=i)["hours"] for i in range(N)])

mean, sd = student.mean(), student.std(ddof=1)
ci = 1.96 * sd / np.sqrt(N)
p_dead = (student < 14).mean()

print(f"Mean battery life : {mean:.2f} h   (95% CI {mean - ci:.2f} – {mean + ci:.2f})")
print(f"Std deviation     : {sd:.2f} h")
print(f"Shortest / longest: {student.min():.1f} h / {student.max():.1f} h")
print(f"P(dies before 10 PM) = {p_dead:.1%}")

plt.figure(figsize=(9, 4.5))
plt.hist(student, bins=30, color="skyblue", edgecolor="k")
plt.axvline(mean, color="red", lw=2, label=f"mean = {mean:.1f} h")
plt.axvline(14, color="black", ls="--", label="10 PM (14 h)")
plt.xlabel("battery life (hours)"); plt.ylabel("number of days")
plt.title("Battery life over 1000 simulated days — Student"); plt.legend()
plt.show()

## 7. Comparing user types
**H₀:** all three user types have the same mean battery life (one-way ANOVA, α = 0.05).

In [ ]:
results = {p: np.array([simulate_day(p, seed=i)["hours"] for i in range(N)]) for p in PROFILES}

for p, x in results.items():
    print(f"{p:<11} mean = {x.mean():5.2f} h   P(dies before 10 PM) = {(x < 14).mean():6.1%}")

f_stat, p_val = stats.f_oneway(*results.values())
print(f"\nANOVA: F = {f_stat:.1f}, p = {p_val:.3g}")
print("→ Reject H0: usage type affects battery life." if p_val < 0.05 else "→ Fail to reject H0.")

plt.figure(figsize=(8, 4.5))
plt.boxplot(list(results.values()))
plt.xticks(range(1, len(results) + 1), list(results.keys()))
plt.ylabel("battery life (hours)"); plt.title("Battery life by user type")
plt.grid(alpha=0.3); plt.show()

## 8. Does Battery Saver mode help?
Same Student, 1000 days with saver **off** vs **on** (same random seeds, so the comparison is fair).
**H₀:** saver mode does not change battery life (two-sample t-test).

In [ ]:
saver = np.array([simulate_day("Student", saver=True, seed=i)["hours"] for i in range(N)])

t_stat, p_t = stats.ttest_ind(saver, student)
gain_min = (saver.mean() - student.mean()) * 60
print(f"Saver OFF: {student.mean():.2f} h   Saver ON: {saver.mean():.2f} h")
print(f"Extra time from Battery Saver ≈ {gain_min:.0f} minutes")
print(f"t-test: t = {t_stat:.2f}, p = {p_t:.3g}")
print("→ Saver mode significantly increases battery life." if p_t < 0.05 else "→ No significant difference.")

plt.figure(figsize=(9, 4.5))
plt.hist(student, bins=30, alpha=0.6, label="Saver OFF")
plt.hist(saver, bins=30, alpha=0.6, label="Saver ON")
plt.xlabel("battery life (hours)"); plt.ylabel("number of days")
plt.title("Effect of Battery Saver mode — Student"); plt.legend(); plt.show()

## 9. Conclusion
1. Battery life is **random**: it changes from day to day depending on which apps are used.
2. Monte Carlo simulation gives the average battery life with a 95% confidence interval, plus the chance the phone dies before night.
3. **Gamers** drain the battery much faster than light users; ANOVA confirms that user type has a significant effect.
4. **Battery Saver** mode adds extra minutes, and the t-test shows whether that gain is significant.

**Limitations / future work:** add screen brightness, battery ageing (capacity loss), charging breaks during the day, or a real usage log from your own phone.